# Speech Recognition

Ziele der heutigen Übung:
- Kennenlernen der SpeechRecognition Bibliothek
- Transkribieren einer Spracheingabe mit der Google Web Speech API
- Verstehen von Erkennerergebnissen eines Spracherkenners
- Anbindung eines großen Sprachmodells (Large-Language Modells) zur Transkription
- Erleben von Herausforderungen der Spracherkennung


## Speech Recognition API

Heute lernen wir eine Bibliothek kennen, welche verschiedene Spracherkenner und online APIs anbindet. Wir verwenden hierfür die Google Speech API als Cloud Service und Vosk als Spracherkenner, welcher lokal mit begrenzten Ressourcen arbeitet.

Installiere folgende Bibliothek: https://pypi.org/project/SpeechRecognition/

In [ ]:
pip install SpeechRecognition

In [126]:
import speech_recognition as sr
print(f"We use speech_recognition version: {sr.__version__}")

We use speech_recognition version: 3.17.0


Um die Eingabe des Mikrofons aufzunehmen und an den Spracherkenner weiter zu leiten, nutzen wir die Bibliothek PyAudio. Installiere PyAudio mit pip. Je nach Betriebssystem kann es erforderlich sein zusätzlieh Pakete mit homebrew (macOS) oder apt (ubuntu) zu installieren. Du findest sicherlich mit einem LLM deiner Wahl oder Google eine Lösung:

In [ ]:
pip install pyaudio

In [127]:
from pyaudio import PyAudio

p = PyAudio()
try:
    print(p.get_default_input_device_info())
except:
    print("No mics available")

{'index': 0, 'structVersion': 2, 'name': 'MacBook Pro-Mikrofon', 'hostApi': 0, 'maxInputChannels': 1, 'maxOutputChannels': 0, 'defaultLowInputLatency': 0.03285416666666666, 'defaultLowOutputLatency': 0.01, 'defaultHighInputLatency': 0.0421875, 'defaultHighOutputLatency': 0.1, 'defaultSampleRate': 48000.0}


## Transkribieren einer sprachlichen Äußerung mit der Google Web Speech API (cloud-basiert)

Folgendes Tutorial gibt gute Hinweise, wie wir die SpeechRecognition Bibliothek nutzen: https://www.simplilearn.com/tutorials/python-tutorial/speech-recognition-in-python 

In [128]:
recognizer = sr.Recognizer()

Erfasse eine sprachliche Äußerung mit dem Mikrofon:

In [129]:
with sr.Microphone() as source:
    print("Speak something...")
    audio_data = recognizer.listen(source)
    print("Audio data recorded.")

Speak something...
Audio data recorded.


Nun schicken wir die erfasste Äußerung an die Google Web Speech API. Sofern die API nicht erreichbar ist, wirft der ```recognizer``` einen ```RequestError```. Sofern in dem übergebenen ```audio_data``` keine sprachliche Äußerung enthalten ist gibt der ```recognizer``` einen ```UnknownValueError``` aus. Prüfe auf beide Exceptions und geben dem Nutzer ein entsprechendes Feedback auf der Konsole aus.

In [130]:
try:
    text = recognizer.recognize_google(audio_data)
    print("You said:", text)
except sr.RequestError:
    print("Error: Could not access Google Web Speech API;")
except sr.UnknownValueError:
    print("Sorry, I do not understand")


You said: hello world Wikipedia


Neben Englisch kann die Google Web Speech API auch Deutsch. Passe den Code entsprechend an, dass Deutsch erkannt wird:

In [131]:
try:
    text = recognizer.recognize_google(audio_data, language='de-DE')
    print("Du hast gesagt: ", text)
except sr.RequestError:
    print("Fehler: Google Web Speech API ist nicht erreichbar")
except sr.UnknownValueError:
    print("Sorry, Ich verstehe die Äußerung nicht")

Du hast gesagt:  hallo Welt wie geht es dir


Wir werden in der nächsten Vorlesung genauer auf die Technik der Spracherkennung schauen. Heute möchten wir uns jedoch bereits mehrer Erkennerergebnisse und die jeweilige Konfidenz anschauen. Mit dem Parameter ```show_all``` gibt der ```recognizer``` mehrere Alternativen als JSON aus. Integriere dies in deine Erkennung (Tipp: für den weiteren Verlauf ist es ratsam dies als Funktion zu implementieren):

In [132]:
import json

def recognize_google(audio_data: any):
    try:
        text = recognizer.recognize_google(audio_data, language='de-DE', show_all=True)
        print(f"[Google] Recognition results: {json.dumps(text, sort_keys=True, indent=4)}")
    except sr.RequestError:
        print("[Google] Fehler: Google Web Speech API ist nicht erreichbar")
    except sr.UnknownValueError:
        print("[Google] Sorry, Ich verstehe die Äußerung nicht")
recognize_google(audio_data)

[Google] Recognition results: {
    "alternative": [
        {
            "confidence": 0.8883912,
            "transcript": "hallo Welt wie geht es dir"
        }
    ],
    "final": true
}


## Spracherkennung ohne Cloud und Internetanbindung

Gerade als Ingenieure haben wir oftmals mit einer fehlenden Internetverbindung in Produktionsumgebungen, begrenzten Hardware-Ressourcen, Datenschutzanforderungen oder was auch immer zu tun, das es verhindert Daten in die Cloud zu schicken. Für solche Fälle möchten wir uns mit https://alphacephei.com/vosk/ einen Spracherkenner anschauen, der einerseits lokal und andererseits auch mit begrenzen Ressourcen zurecht kommt (z.B. Raspberry PI). Installiere die Bibliothek mit ```pip```

In [ ]:
pip install vosk

Vosk unterstützt verschiedene Sprachen und bietet Sprachmodelle in unterschiedlicher Größe an: siehe hierzu https://alphacephei.com/vosk/models. Starte mit einem kleinen Modell (vosk-model-small-**), lade dieses herunter. 

Für den Download können wir das Shellscript ```sprc``` nutzen. Wir können dies direkt in einer Zelle dieses Jupyter Notebooks machen, indem wir eine Code Zelle erstellen und als Sprache Shell Script auswählen. ```! sprc download vosk --url <ModellURL einfügen>```:

In [ ]:
! sprc download vosk --url https://alphacephei.com/vosk/models/vosk-model-small-de-0.15.zip

Nutze die Funktion ```recognizer.recognize_vosk```, um deine Äußerung zu transkribieren. Auch hier ist es hilfreich dies als Funktion zu implementieren...


In [ ]:
with sr.Microphone() as source:
    print("Speak something...")
    audio_data = recognizer.listen(source)
    print("Audio data recorded.")

In [133]:
def recognize_vosk(audio_data: any):
    try:
        text = recognizer.recognize_vosk(audio_data)
        print("[Vosk] You said:", text)
    except sr.RequestError:
        print("[Vosk] Error: Request Error;")
    except sr.UnknownValueError:
        print("[Vosk] Sorry, I do not understand")
recognize_vosk(audio_data)

LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=10 max-active=3000 lattice-beam=2
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from /Users/winkelmannsv/git/dialogsystems/.venv/lib/python3.13/site-packages/speech_recognition/models/vosk/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from /Users/winkelmannsv/git/dialogsystems/.venv/lib/python3.13/site-packages/speech_recognition/models/vosk/graph/HCLr.fst /Users/winkelmannsv/git/dialogsystems/.venv/lib/python3.13/site-packages/speech_recognition/models/vosk/g

[Vosk] You said: hallo welt wie geht es dir


## Herausforderungen der Spracherkennung

Wir haben nun zwei Erkenner angebunden, einerseits ein großes Modell in der Cloud und andererseits ein kleines Modell, welches lokal erkennt. Doch wie gut funktionieren diese? Und wo sind die Unterschiede? Nehme dir etwas Zeit und versuche verschiedene Äußerungen aus, variiere dabei zum Beispiel:
- Anwendungsfälle
- Namen von Personen, Städten, Restaurants, ...
- Sprechgeschwindigkeit
- Lautstärke
- Deutlich vs. undeutliche Aussprache
- Dialekt (kann Google Fränkisch?)
- Kommandos vs. natürlichsprachlich
- Lange vs. kurze Äußerungen
- Englische Begriffe in einem deutschen Satz (oder anders rum)
- Umgangssprache

Speichere dir hierfür die mit dem Mikrofon aufgenommenen Daten in eine Variable ```audio_data``` und gebe diese Daten an ```recognize_google(audio_data, ..``` und ```recognize_vosk(audio_data, ..```. Hier ist es nun praktisch auf die zuvor erstellen Funktionen zurückzugreifen ;-)

In [ ]:
with sr.Microphone() as source:
    print("Speak something...")
    audio_data = recognizer.listen(source)
    print("Audio data recorded.")
    
    recognize_google(audio_data)
    recognize_vosk(audio_data)

Speak something...
Audio data recorded.
[Google] Recognition results: {
    "alternative": [
        {
            "confidence": 0.8883912,
            "transcript": "hallo Welt wie geht es dir"
        }
    ],
    "final": true
}


LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=10 max-active=3000 lattice-beam=2
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from /Users/winkelmannsv/git/dialogsystems/.venv/lib/python3.13/site-packages/speech_recognition/models/vosk/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from /Users/winkelmannsv/git/dialogsystems/.venv/lib/python3.13/site-packages/speech_recognition/models/vosk/graph/HCLr.fst /Users/winkelmannsv/git/dialogsystems/.venv/lib/python3.13/site-packages/speech_recognition/models/vosk/g

[Vosk] You said: hallo welt wie geht es dir


## Vertiefung: Transkription und Übersetzung mit Faster Whisper

Neben Vosk und Google kann die SpechRecognition Library auch mit diversen anderen Sprachanbietern umgehen. Im entsprechenden GitHub Repo findest du eine Auflistung:
https://github.com/Uberi/speech_recognition/

Bei den Cloud Anbietern benötigt man in der Regel einen (kostenpflichtigen) API-Key. Interessant sind die OpenAI Whisper Modelle, die zusätzlich lokal laufen und recht vernünftige Ergebnisse liefern (eine Übersicht zu aktuellen Benchmarks liefert: https://huggingface.co/spaces/hf-audio/open_asr_leaderboard). Wegen der Modellgröße und Rechenkapazität möchten wir im folgenden ein faster_whisper Modell nutzen. Von der Implementierung sind größere Modelle jedoch identisch.

Installiere als erstes die notwendigen Packete:

In [ ]:
pip install faster_whisper; soundfile

Dann können wir auch schon ```recognize_faster_whisper(audio_data)``` nutzen, um unser Audiofile mit dem Faster-Whsiper base Modell zu transkribieren:

In [135]:
recognizer.recognize_faster_whisper(audio_data)

[2026-10-15 09:54:36.191] [ctranslate2] [thread 920025] [warning] The compute type inferred from the saved model is float16, but the target device or backend do not support efficient float16 computation. The model weights have been automatically converted to use the float32 compute type instead.


' Hallo Welt, wie geht es dir?'

Mit dem Parameter ```show_dict=True``` bekommen wir noch einige interessanten Zusatzinformationen:

In [136]:
result = recognizer.recognize_faster_whisper(audio_data, show_dict=True)
result

[2026-10-15 09:54:43.483] [ctranslate2] [thread 920025] [warning] The compute type inferred from the saved model is float16, but the target device or backend do not support efficient float16 computation. The model weights have been automatically converted to use the float32 compute type instead.


{'text': ' Hallo Welt, wie geht es dir?',
 'segments': [Segment(id=1, seek=0, start=0.0, end=2.0, text=' Hallo Welt, wie geht es dir?', tokens=[50364, 21242, 14761, 11, 3355, 7095, 785, 4746, 30, 50464], avg_logprob=-0.2737105163660916, compression_ratio=0.7777777777777778, no_speech_prob=0.010807869955897331, words=None, temperature=0.0)],
 'language': 'de'}

In [137]:
print("Die Wahrscheinlichkeit, dass keine Sprache im Audiofile ist beträgt: ", result['segments'][0].no_speech_prob)

Die Wahrscheinlichkeit, dass keine Sprache im Audiofile ist beträgt:  0.010807869955897331


Ebenso können wir über die ```init_options``` und ```transcribe_options``` weitere Parameter mitschicken und somit die Transkription von Whisper steuern. Viele der Parameter werden wir im Kapitel LLMs genauer betrachten. 

In [138]:
from speech_recognition.recognizers.whisper_local.faster_whisper import InitOptionalParameters
from speech_recognition.recognizers.whisper_local.faster_whisper import TranscribeOptionalParameters

# see https://github.com/SYSTRAN/faster-whisper/blob/v1.1.0/faster_whisper/transcribe.py#L575
init_params = InitOptionalParameters()
init_params['device'] = 'cpu'  # gpu
init_params['num_workers'] = 5

# see https://github.com/SYSTRAN/faster-whisper/blob/v1.1.0/faster_whisper/transcribe.py#L692
transcribe_params = TranscribeOptionalParameters()
transcribe_params['task'] = 'translate'  # transcribe
transcribe_params['temperature'] = [0.0]
transcribe_params['word_timestamps'] = True

In [139]:
result = recognizer.recognize_faster_whisper(audio_data, show_dict=True, init_options=init_params, **transcribe_params) 
result

[2026-10-15 09:55:12.044] [ctranslate2] [thread 920025] [warning] The compute type inferred from the saved model is float16, but the target device or backend do not support efficient float16 computation. The model weights have been automatically converted to use the float32 compute type instead.


{'text': ' Hello world, how are you?',
 'segments': [Segment(id=1, seek=0, start=np.float64(0.0), end=np.float64(1.84), text=' Hello world, how are you?', tokens=[50364, 2425, 1002, 11, 577, 366, 291, 30, 50464], avg_logprob=-0.4697524309158325, compression_ratio=0.7575757575757576, no_speech_prob=0.010807869955897331, words=[Word(start=np.float64(0.0), end=np.float64(0.66), word=' Hello', probability=np.float64(0.6772043704986572)), Word(start=np.float64(0.66), end=np.float64(0.94), word=' world,', probability=np.float64(0.5164812207221985)), Word(start=np.float64(1.3), end=np.float64(1.42), word=' how', probability=np.float64(0.766306459903717)), Word(start=np.float64(1.42), end=np.float64(1.56), word=' are', probability=np.float64(0.47448891401290894)), Word(start=np.float64(1.56), end=np.float64(1.84), word=' you?', probability=np.float64(0.9890312552452087))], temperature=0.0)],
 'language': 'de'}

In [140]:
transcribe_params['temperature'] = [1.0]
result = recognizer.recognize_faster_whisper(audio_data, show_dict=True, init_options=init_params, **transcribe_params) 
result

[2026-10-15 09:55:38.087] [ctranslate2] [thread 920025] [warning] The compute type inferred from the saved model is float16, but the target device or backend do not support efficient float16 computation. The model weights have been automatically converted to use the float32 compute type instead.


{'text': ' How are you?',
 'segments': [Segment(id=1, seek=0, start=np.float64(0.0), end=np.float64(1.84), text=' How are you?', tokens=[50364, 1012, 366, 291, 30, 50464], avg_logprob=-0.9177251202719552, compression_ratio=0.6, no_speech_prob=0.010807869955897331, words=[Word(start=np.float64(0.0), end=np.float64(0.82), word=' How', probability=np.float64(0.059943076223134995)), Word(start=np.float64(0.82), end=np.float64(1.6), word=' are', probability=np.float64(0.17207039892673492)), Word(start=np.float64(1.6), end=np.float64(1.84), word=' you?', probability=np.float64(0.9687426686286926))], temperature=1.0)],
 'language': 'de'}